# 101 Spark basics

The goal of this lab is to get familiar with Spark programming.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

sc

<SparkContext master=local appName=Local Spark>

## 101-1 Spark warm-up

Load the ```capra``` and ```divinacommedia``` datasets and try the following actions:
- Show their content (```collect```)
- Count their rows (```count```)
- Split phrases into words (```map``` or ```flatMap```; what’s the difference?)
- Check the results (remember: evaluation is lazy)
- Try the ```toDebugString``` function to check the execution plan
    - In PySpark, use ```toDebugString().decode("unicode_escape")```

In [13]:
#rddCapra = sc.textFile("../../../../datasets/capra.txt")
#rddDC = sc.textFile("../../../../datasets/divinacommedia.txt")
rddCapra = sc.textFile("file:///home/jovyan/work/datasets/capra.txt")
rddDC = sc.textFile("file:///home/jovyan/work/datasets/divinacommedia.txt")

print(rddCapra.collect())

['sopra la panca la capra campa', 'sotto la panca la capra crepa']


In [11]:
import os

print("Directory corrente:", os.getcwd())
print("File capra:", os.path.abspath("datasets/capra.txt"))
print("Esiste:", os.path.exists("datasets/capra.txt"))

Directory corrente: /home/jovyan
File capra: /home/jovyan/datasets/capra.txt
Esiste: False


In [15]:
import os

for root, dirs, files in os.walk("."):
    if "capra.txt" in files:
        print(os.path.abspath(os.path.join(root, "capra.txt")))

/home/jovyan/work/datasets/capra.txt


In [31]:
print(rddCapra.collect())

print(rddCapra.map( lambda x: x.split(" ")).collect())
print(rddCapra.flatMap(lambda x: x.split(" ")).collect())
#print(rddDC.map( lambda x: x.split(" ")).collect())
#print(rddDC.flatMap( lambda x: x.split(" ")).collect())

['sopra la panca la capra campa', 'sotto la panca la capra crepa']
[['sopra', 'la', 'panca', 'la', 'capra', 'campa'], ['sotto', 'la', 'panca', 'la', 'capra', 'crepa']]
['sopra', 'la', 'panca', 'la', 'capra', 'campa', 'sotto', 'la', 'panca', 'la', 'capra', 'crepa']


## 101-2 Basic Spark jobs

Implement on Spark the following jobs and test them on both capra and divinacommedia datasets.

- **Word count**: count the number of occurrences of each word
  - Result: (sopra, 1), (la, 4), …
- **Word length count**: count the number of occurrences of words of given lengths
  - Result: (2, 4), (5, 8)
- Count the average length of words given their first letter (i.e., words that begin with "s" have an average length of 5)
  - Result: (s, 5), (l, 2), …
- Return the inverted index of words (i.e., for each word, list the numbers of lines in which they appear)
  - Result: (sopra, (0)), (la, (0, 1)), ...

Also, check how sorting works and try to sort key-value RDDs by descending values.

In [17]:
rddCapraWords = rddCapra.flatMap(lambda x: x.split(" "))
rddCapraCount = rddCapraWords.map(lambda x: (x, 1)).reduceByKey(lambda a, b: a + b)
print(rddCapraCount.collect())

[('sopra', 1), ('la', 4), ('panca', 2), ('capra', 2), ('campa', 1), ('sotto', 1), ('crepa', 1)]


In [18]:
rddCapraLengthCount = rddCapraWords.map(lambda x: (len(x), 1)).reduceByKey(lambda a, b: a + b)
print(rddCapraLengthCount.collect())

[(5, 8), (2, 4)]


In [ ]:
rddCapraAvarage = rddCapraWords.\
                    map(lambda x: (x[0], (len(x), 1))).\
                    reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).\
                    map(lambda x: (x[0], x[1][0] / x[1][1]))
print(rddCapraAvarage.collect())

[('s', 5.0), ('l', 2.0), ('p', 5.0), ('c', 5.0)]


In [ ]:
#my solution:
rddCapra.\
    map(lambda x: x.split(" ")).\
    zipWithIndex().\
    flatMap(lambda x: [(elem, [x[1]]) for elem in x[0]]).\
    reduceByKey(lambda a, b: a + b).\
    collect()

[('sopra', [0]),
 ('la', [0, 0, 1, 1]),
 ('panca', [0, 1]),
 ('capra', [0, 1]),
 ('campa', [0]),
 ('sotto', [1]),
 ('crepa', [1])]

In [ ]:
#prof
rddCapra.\
    flatMap(lambda x: x.split(" ")).\
    zipWithIndex().\
    groupByKey().\
    mapValues(lambda v: set(v)).\
    collect()

[('sopra', {0}),
 ('la', {1, 3, 7, 9}),
 ('panca', {2, 8}),
 ('capra', {4, 10}),
 ('campa', {5}),
 ('sotto', {6}),
 ('crepa', {11})]

In [ ]:
#prof
rddCapra.\
    zipWithIndex().\
    map(lambda kv: (kv[1], kv[0])).\
    flatMapValues(lambda line: line.split(" ")).\
    map(lambda kv: (kv[1], kv[0])).\
    groupByKey().\
    mapValues(lambda v: set(v)).\
    collect()

[('sopra', {0}),
 ('la', {0, 1}),
 ('panca', {0, 1}),
 ('capra', {0, 1}),
 ('campa', {0}),
 ('sotto', {1}),
 ('crepa', {1})]

## 101-3 Extra Spark jobs

Implement the following job.

- Co-occurrence count: count the number of co-occurrences in the text. A co-occurrence is defined as "two distinct words appearing in the same line".
  - In the first line of the *capra* dataset, co-occurrences are:
     - (sopra, la), (sopra, panca), (sopra, capra), (sopra, campa)
     - (la, sopra), (la, panca), (la, capra), (la, campa) 
     - (panca, sopra), (panca, la), (panca, capra), (panca, campa)
     - (capra, sopra), (capra, la), (capra, panca), (capra, campa)
     - (campa, sopra), (campa, la), (campa, panca), (campa, capra)